# Disease ↔ Gene Relation-Wise Merge

Merges Disease–Gene triples from DRKG, PrimeKG, PharmKG, Hetionet, TARKG, iBKH,
and EvoAGE; resolves disease names via DO/MESH and gene names via NCBI;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [48]:
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_GENE/ALL_DISEASE_GENE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Gene — NCBI and ENSEMBL

In [2]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_ALL_GENE_SYM_DES    = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")

NCBI gene table: 193,505 rows


### 1b. Disease — DO, MedGen, MESH

In [3]:
# Disease Ontology
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [4]:
# MedGen: source_id → preferred name; MONDO → MESH CUI
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [5]:
# MESH combined and supplementary: ID → Name
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


## 2. Helper — Resolve Disease Head Node

In [6]:
def resolve_disease_node(df, node):
    """Map MONDO→MESH, derive detail_name, assign id_is for head or tail."""
    df[node] = df[node].map(MONDO_2_MESH_dict).fillna(df[node])
    df = df[~df[node].astype(str).str.startswith('MONDO')].copy()
    df[f'{node}_detail_name'] = df[node].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df[f'{node}_id_is'] = np.where(
        df[node].isna(), None,
        np.where(df[node].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. DRKG

In [7]:
DRKG_Disease_Gene = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Disease_Gene.csv')
DRKG_Disease_Gene.columns = DRKG_Disease_Gene.columns.str.lower()
print(f"DRKG:     {len(DRKG_Disease_Gene):,} rows | columns: {list(DRKG_Disease_Gene.columns)}")
DRKG_Disease_Gene['kg_type'] = 'Generalised'
DRKG_Disease_Gene['kg_source'] = 'DRKG'
DRKG_Disease_Gene['species'] = 'Homo species'

DRKG_Disease_Gene.head(2)

DRKG:     28,388 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'tail_detail_name', 'tail_ens', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,tail_ens,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,SARS-CoV2 E,Disease_Gene,AP3B1,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,SARS-CoV2 E,adaptor related protein complex 3 subunit beta 1,ENSG00000132842,MESH,NCBI_ID,Disease::SARS-CoV2 E,Gene::8546,Generalised,Homo species
1,SARS-CoV2 E,Disease_Gene,BRD4,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,SARS-CoV2 E,bromodomain containing 4,ENSG00000141867,MESH,NCBI_ID,Disease::SARS-CoV2 E,Gene::23476,Generalised,Homo species


### 3b. PrimeKG

In [8]:
PrimeKG_Disease_Gene = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Disease_Gene.csv')
PrimeKG_Disease_Gene.columns = PrimeKG_Disease_Gene.columns.str.lower()
PrimeKG_Disease_Gene.rename(columns={'head_do_name': 'head_detail_name'}, inplace=True)
PrimeKG_Disease_Gene = resolve_disease_node(PrimeKG_Disease_Gene, 'head')
print(f"PrimeKG:  {len(PrimeKG_Disease_Gene):,} rows")

PrimeKG_Disease_Gene['kg_type'] = 'Generalised'
PrimeKG_Disease_Gene['species'] = 'Homo species'

PrimeKG_Disease_Gene.head(2)

PrimeKG:  70,366 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,head_do_alt_ids,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,C0036341,Disease_Gene,A1BG,Disease,associated with,Gene,PrimeKG,MESH,NCBI_Symbol,Schizophrenia,NaN,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised,Homo species
1,C0036341,Disease_Gene,ABCA1,Disease,associated with,Gene,PrimeKG,MESH,NCBI_Symbol,Schizophrenia,NaN,ABCA1,ATP binding cassette subfamily A member 1,ENSG00000165029,Generalised,Homo species


### 3c. PharmKG

In [9]:
PharmKG_Disease_Gene = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Disease_Gene.csv')
PharmKG_Disease_Gene.columns = PharmKG_Disease_Gene.columns.str.lower()
PharmKG_Disease_Gene.rename(columns={'head_id': 'head_detail_name'}, inplace=True)
PharmKG_Disease_Gene['tail_detail_name'] = PharmKG_Disease_Gene['tail'].map(NCBI_ALL_GENE_SYM_DES)
print(f"PharmKG:  {len(PharmKG_Disease_Gene):,} rows")


PharmKG_Disease_Gene['kg_type'] = 'Generalised'
PharmKG_Disease_Gene['species'] = 'Homo species'


PharmKG_Disease_Gene.head(2)

PharmKG:  22 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_alias_mapped,head_id_is,tail_id_is,head_orignal,pubmed_id,sentence_tokenized,kg_type,species
0,DOID:768,Disease_Gene,AR,disease,Rg,gene,PharmKG,androgen receptor,AREG,DOID,NCBI_ID,retinoblastoma,NaN,NaN,Generalised,Homo species
1,DOID:2121,Disease_Gene,EDA,disease,H,gene,PharmKG,ectodysplasin A,EDA,DOID,NCBI_ID,ectodermal dysplasia,"15663448.0, 8071953.0, 21652647.0, 21410470.0,...",'X-linked hypohidrotic ectodermal_dysplasia -L...,Generalised,Homo species


### 3d. Hetionet

In [10]:
Hetionet_Disease_Gene = pd.read_csv(PROC_DIR + 'Hetionet/Disease_Gene_Hetionet.csv')
Hetionet_Disease_Gene.columns = Hetionet_Disease_Gene.columns.str.lower()
Hetionet_Disease_Gene.rename(columns={'head_name': 'head_detail_name'}, inplace=True)
print(f"Hetionet: {len(Hetionet_Disease_Gene):,} rows")

Hetionet_Disease_Gene['kg_type'] = 'Generalised'

Hetionet_Disease_Gene['species'] = 'Homo species'


Hetionet_Disease_Gene.head(2)

Hetionet: 27,936 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_name,tail_detail_name,kg_type,species
0,DOID:986,Disease_Gene,ERC2,Disease,Disease–downregulates–Gene,Gene,Hetionet,DOID,NCBI_ID,alopecia areata,26059,ELKS/RAB6-interacting/CAST family member 2,Generalised,Homo species
1,DOID:9206,Disease_Gene,MTX2,Disease,Disease–downregulates–Gene,Gene,Hetionet,DOID,NCBI_ID,Barrett's esophagus,10651,metaxin 2,Generalised,Homo species


### 3e. TARKG

In [11]:
TARKG_Disease_Gene = pd.read_csv(PROC_DIR + 'TARKG/Disease_Gene_TARKG.csv')
TARKG_Disease_Gene.columns = TARKG_Disease_Gene.columns.str.lower()
print(f"TARKG:    {len(TARKG_Disease_Gene):,} rows")

TARKG_Disease_Gene['kg_type'] = 'Generalised'
TARKG_Disease_Gene['species'] = 'Homo species'

TARKG_Disease_Gene.head(2)

TARKG:    1,303,254 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,DOID:0001816,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,angiosarcoma,DNA topoisomerase II alpha,DOID,NCBI_ID,addKG-14368360,addKG,0,Generalised,Homo species
1,D003141,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,Communicable Diseases,DNA topoisomerase II alpha,MESH,NCBI_ID,addKG-14303082,addKG,0,Generalised,Homo species


### 3f. iBKH

Fix swapped `tail_detail_name` columns.

In [12]:
iBKH_Disease_Gene = pd.read_csv(PROC_DIR + 'iBKH/Disease_Gene_ibkh.csv')
iBKH_Disease_Gene.columns = iBKH_Disease_Gene.columns.str.lower()
# iBKH_Disease_Gene.drop(columns=['tail_detail_name'], inplace=True)
# iBKH_Disease_Gene.rename(columns={'tail_detail_name.1': 'tail_detail_name'}, inplace=True)
print(f"iBKH:     {len(iBKH_Disease_Gene):,} rows")

iBKH_Disease_Gene['kg_type'] = 'Generalised'
iBKH_Disease_Gene['species'] = 'Homo species'
iBKH_Disease_Gene.head(2)

iBKH:     12,323,320 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,DOID:263,Disease_Gene,IFNA1,Disease,Gene,iBKH,kidney cancer,interferon alpha 1,DOID,NCBI_ID,Generalised,Homo species
1,DOID:1909,Disease_Gene,VCAN,Disease,Gene,iBKH,melanoma,versican,DOID,NCBI_ID,Generalised,Homo species


### 3g. hald

In [13]:
hald = pd.read_csv(PROC_DIR + 'hald/Disease_Gene_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald:   {len(hald):,} rows")

hald['kg_type'] = 'Aging'
hald['species'] = 'Homo species'

hald.head(2)

hald:   6,813 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,D065626,Disease_Gene,GPT,Disease,associated,Gene,HALD_KG,Non-alcoholic Fatty Liver Disease,glutamic--pyruvic transaminase,MESH,NCBI_ID,Aging,Homo species
1,D000544,Disease_Gene,C9orf72,Disease,genotype,Gene,HALD_KG,Alzheimer Disease,C9orf72-SMCR8 complex subunit,MESH,NCBI_ID,Aging,Homo species


## 4. Check and Fix Duplicate Columns

In [15]:
SOURCE_DFS = [
    ('DRKG_Disease_Gene',     DRKG_Disease_Gene),
    ('PrimeKG_Disease_Gene',  PrimeKG_Disease_Gene),
    ('PharmKG_Disease_Gene',  PharmKG_Disease_Gene),
    ('Hetionet_Disease_Gene', Hetionet_Disease_Gene),
    ('TARKG_Disease_Gene',    TARKG_Disease_Gene),
    ('iBKH_Disease_Gene',     iBKH_Disease_Gene),
    ('hald',                hald),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[DRKG_Disease_Gene] ✓ no duplicates
[PrimeKG_Disease_Gene] ✓ no duplicates
[PharmKG_Disease_Gene] ✓ no duplicates
[Hetionet_Disease_Gene] ✓ no duplicates
[TARKG_Disease_Gene] ✓ no duplicates
[iBKH_Disease_Gene] ✓ no duplicates
[hald] ✓ no duplicates


In [16]:
# DRKG_Disease_Gene     = DRKG_Disease_Gene.loc[:,     ~DRKG_Disease_Gene.columns.duplicated(keep='first')]
# PrimeKG_Disease_Gene  = PrimeKG_Disease_Gene.loc[:,  ~PrimeKG_Disease_Gene.columns.duplicated(keep='first')]
# PharmKG_Disease_Gene  = PharmKG_Disease_Gene.loc[:,  ~PharmKG_Disease_Gene.columns.duplicated(keep='first')]
# Hetionet_Disease_Gene = Hetionet_Disease_Gene.loc[:, ~Hetionet_Disease_Gene.columns.duplicated(keep='first')]
# TARKG_Disease_Gene    = TARKG_Disease_Gene.loc[:,    ~TARKG_Disease_Gene.columns.duplicated(keep='first')]
# iBKH_Disease_Gene     = iBKH_Disease_Gene.loc[:,     ~iBKH_Disease_Gene.columns.duplicated(keep='first')]
# EvoAGE                = EvoAGE.loc[:,                ~EvoAGE.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

## 5. Align Schemas and Concatenate

In [17]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 13,760,099 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,SARS-CoV2 E,Disease_Gene,AP3B1,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 E,adaptor related protein complex 3 subunit beta 1,Generalised,Homo species
1,SARS-CoV2 E,Disease_Gene,BRD4,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 E,bromodomain containing 4,Generalised,Homo species


## 6. Standardise Fixed-Value Columns

In [18]:
final_df['head_type'] = 'Disease'
final_df['tail_type'] = 'Gene'
final_df['relation']  = 'Disease_Gene'

# Normalise id_is artefacts from individual sources
final_df['head_id_is'] = final_df['head_id_is'].str.replace('DO_ID', 'DOID', regex=False)
final_df['tail_id_is'] = final_df['tail_id_is'].str.replace('NCBI_Symbol', 'NCBI_ID', regex=False)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Disease_Gene'}
Unique head_id_is: {'DOID', 'MESH'}
Unique tail_id_is: {'NCBI_ID'}
Unique kg_source: {'PrimeKG', 'PharmKG', 'HALD_KG', 'iBKH', 'TARKG', 'Hetionet', 'DRKG'}


## 7. Deduplicate

In [19]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})

print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 13,054,269 rows


## 8. Drop Unresolvable Rows and Add Schema Columns

In [20]:
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing tail_detail_name")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Dropped 4 rows with missing tail_detail_name
Final shape: (13054265, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0000744,Disease_Gene,APOB,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,apolipoprotein B,Generalised,Homo species
1,C0000744,Disease_Gene,TRNP,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,tRNA-Pro,Generalised,Homo species
2,C0000809,Disease_Gene,CREB5,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,cAMP responsive element binding protein 5,Generalised,Homo species
3,C0000809,Disease_Gene,SYCP3,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,synaptonemal complex protein 3,Generalised,Homo species
4,C0000832,Disease_Gene,BHMT,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Placental abruption,betaine--homocysteine S-methyltransferase,Generalised,Homo species


## 9. QC — NaN Check

In [21]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,11853283,0,11853283
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,8,0,8


## 10. Save Output

In [35]:
OUT_PATH
final_df_dedup['species'] = 'Homo sapiens'

In [22]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,C0000744,Disease_Gene,APOB,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,apolipoprotein B,Generalised,Homo species,Homo sapiens,Homo sapiens
1,C0000744,Disease_Gene,TRNP,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,tRNA-Pro,Generalised,Homo species,Homo sapiens,Homo sapiens
2,C0000809,Disease_Gene,CREB5,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,cAMP responsive element binding protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
3,C0000809,Disease_Gene,SYCP3,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,synaptonemal complex protein 3,Generalised,Homo species,Homo sapiens,Homo sapiens
4,C0000832,Disease_Gene,BHMT,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Placental abruption,betaine--homocysteine S-methyltransferase,Generalised,Homo species,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13054260,SARS-CoV2 orf9c,Disease_Gene,TMED5,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane p24 trafficking protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
13054261,SARS-CoV2 orf9c,Disease_Gene,TMEM39B,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 39B,Generalised,Homo species,Homo sapiens,Homo sapiens
13054262,SARS-CoV2 orf9c,Disease_Gene,TMEM97,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 97,Generalised,Homo species,Homo sapiens,Homo sapiens
13054263,SARS-CoV2 orf9c,Disease_Gene,UBXN8,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,UBX domain protein 8,Generalised,Homo species,Homo sapiens,Homo sapiens


In [23]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 13,054,265 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_GENE/ALL_DISEASE_GENE.csv


# Final Mapping

In [49]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [53]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,C0000744,Disease_Gene,APOB,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,apolipoprotein B,Generalised,Homo species,Homo sapiens,Homo sapiens
1,C0000744,Disease_Gene,TRNP,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,tRNA-Pro,Generalised,Homo species,Homo sapiens,Homo sapiens
2,C0000809,Disease_Gene,CREB5,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,cAMP responsive element binding protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
3,C0000809,Disease_Gene,SYCP3,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,synaptonemal complex protein 3,Generalised,Homo species,Homo sapiens,Homo sapiens
4,DOID:9667,Disease_Gene,BHMT,Disease,associated with,Gene,PrimeKG,DOID,NCBI_ID,Placental abruption,betaine--homocysteine S-methyltransferase,Generalised,Homo species,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13054260,SARS-CoV2 orf9c,Disease_Gene,TMED5,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane p24 trafficking protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
13054261,SARS-CoV2 orf9c,Disease_Gene,TMEM39B,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 39B,Generalised,Homo species,Homo sapiens,Homo sapiens
13054262,SARS-CoV2 orf9c,Disease_Gene,TMEM97,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 97,Generalised,Homo species,Homo sapiens,Homo sapiens
13054263,SARS-CoV2 orf9c,Disease_Gene,UBXN8,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,UBX domain protein 8,Generalised,Homo species,Homo sapiens,Homo sapiens


In [55]:
final_file = final_file[~final_file['head'].isna()]
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,C0000744,Disease_Gene,APOB,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,apolipoprotein B,Generalised,Homo species,Homo sapiens,Homo sapiens
1,C0000744,Disease_Gene,TRNP,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Abetalipoproteinaemia,tRNA-Pro,Generalised,Homo species,Homo sapiens,Homo sapiens
2,C0000809,Disease_Gene,CREB5,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,cAMP responsive element binding protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
3,C0000809,Disease_Gene,SYCP3,Disease,associated with,Gene,PrimeKG,MESH,NCBI_ID,Habitual spontaneous abortion,synaptonemal complex protein 3,Generalised,Homo species,Homo sapiens,Homo sapiens
4,DOID:9667,Disease_Gene,BHMT,Disease,associated with,Gene,PrimeKG,DOID,NCBI_ID,Placental abruption,betaine--homocysteine S-methyltransferase,Generalised,Homo species,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13054260,SARS-CoV2 orf9c,Disease_Gene,TMED5,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane p24 trafficking protein 5,Generalised,Homo species,Homo sapiens,Homo sapiens
13054261,SARS-CoV2 orf9c,Disease_Gene,TMEM39B,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 39B,Generalised,Homo species,Homo sapiens,Homo sapiens
13054262,SARS-CoV2 orf9c,Disease_Gene,TMEM97,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,transmembrane protein 97,Generalised,Homo species,Homo sapiens,Homo sapiens
13054263,SARS-CoV2 orf9c,Disease_Gene,UBXN8,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,DRKG,MESH,NCBI_ID,SARS-CoV2 orf9c,UBX domain protein 8,Generalised,Homo species,Homo sapiens,Homo sapiens


In [56]:
final_file[final_file['tail'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species


In [57]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 13,054,257 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_GENE/ALL_DISEASE_GENE.csv
